# Ingestión del archivo `person.json`

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

## 1. Leer el archivo JSON usando `DataFrameReader` de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

person_schema = StructType([
    StructField("personId", IntegerType()),
    StructField("personName", StructType([
        StructField("forename", StringType()),
        StructField("surname", StringType()),
    ]))
])


persons_df = (spark.read 
    .schema(person_schema)
    .json(f"{bronze_folder_path}/{v_file_date}/person.json")
)
display(persons_df)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:440)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:80)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:348)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Attr

## 2. Agregar la columna `name` a partir de la concatenación de `forename` y `surname`

In [0]:
from pyspark.sql.functions import concat, lit

persons_with_columns_df = persons_df.withColumn("name", concat(persons_df.personName.forename, persons_df.personName.surname))
display(persons_with_columns_df)

personId,personName,name
1,"List(George , Lucas)",George Lucas
2,"List(Mark , Hamill)",Mark Hamill
3,"List(Harrison , Ford)",Harrison Ford
4,"List(Carrie , Fisher)",Carrie Fisher
5,"List(Peter , Cushing)",Peter Cushing
6,"List(Anthony , Daniels)",Anthony Daniels
7,"List(Andrew , Stanton)",Andrew Stanton
8,"List(Lee , Unkrich)",Lee Unkrich
9,"List(Graham , Walters)",Graham Walters
10,"List(Bob , Peterson)",Bob Peterson


## 3. Eliminar las columnas no deseadas del DataFrame

In [0]:
persons_dropped_df = persons_with_columns_df.drop("personName")

## 4. Cambiar el nombre de las columnas según lo requerido

In [0]:
persons_renamed_df = (persons_dropped_df
    .withColumnRenamed("personId", "person_id")
)

## 5. Agregar las columnas `ingestion_date` y `environment` al DateFrame

In [0]:
from pyspark.sql.functions import current_timestamp

persons_final_df = add_ingestion_date(persons_renamed_df).withColumn("enviroment", lit(v_environment)).withColumn("file_date", lit(v_file_date))


## 5. Escribir datos en el datalake en formato `Parquet`

In [0]:
merge_delta_lake( persons_final_df, "movie_silver", "persons", "tgt.person_id = src.person_id AND tgt.file_date = src.file_date", "file_date" )

In [0]:
%sql
SELECT * FROM movie_silver.persons

path,name,size,modificationTime
abfss://silver@moviehistory4.dfs.core.windows.net/persons/_SUCCESS,_SUCCESS,0,1789059241000
abfss://silver@moviehistory4.dfs.core.windows.net/persons/_committed_7568689033949375427,_committed_7568689033949375427,221,1789059152000
abfss://silver@moviehistory4.dfs.core.windows.net/persons/_committed_7858477622220500853,_committed_7858477622220500853,429,1789059241000
abfss://silver@moviehistory4.dfs.core.windows.net/persons/_started_7568689033949375427,_started_7568689033949375427,0,1789059149000
abfss://silver@moviehistory4.dfs.core.windows.net/persons/_started_7858477622220500853,_started_7858477622220500853,0,1789059238000
abfss://silver@moviehistory4.dfs.core.windows.net/persons/part-00000-tid-7858477622220500853-c84c4f20-b49c-4f1b-9bf7-841ae35333da-15-1-c000.snappy.parquet,part-00000-tid-7858477622220500853-c84c4f20-b49c-4f1b-9bf7-841ae35333da-15-1-c000.snappy.parquet,912366,1789059240000
abfss://silver@moviehistory4.dfs.core.windows.net/persons/part-00001-tid-7858477622220500853-c84c4f20-b49c-4f1b-9bf7-841ae35333da-16-1-c000.snappy.parquet,part-00001-tid-7858477622220500853-c84c4f20-b49c-4f1b-9bf7-841ae35333da-16-1-c000.snappy.parquet,830525,1789059239000
